In [1]:
import pandas as pd
import chardet
import logging
from pathlib import Path
from datetime import date
import re

# I use logging instead of print so output is properly timestamped and levelled.
# In production on Databricks this integrates directly with the cluster log driver.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# Paths are defined relative to the notebook so they work on any machine.
# To have this in Azure, these would be replaced with the corresponding paths.
BASE_DIR      = Path("..")
RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "RawData.csv"
COLUMNS_PATH  = BASE_DIR / "data" / "raw" / "Columns.csv"

# I follow the Raw, Prepared, Staging naming convention for each layer.
RAW_DIR      = BASE_DIR / "data" / "raw_layer"
PREPARED_DIR = BASE_DIR / "data" / "prepared_layer"
STAGING_DIR  = BASE_DIR / "data" / "staging_layer"

for d in [RAW_DIR, PREPARED_DIR, STAGING_DIR]:
    d.mkdir(parents=True, exist_ok=True)

INGESTION_DATE = date.today().isoformat()

logger.info("Ingestion date : %s", INGESTION_DATE)
logger.info("Raw data path  : %s", RAW_DATA_PATH.resolve())

2026-03-10 16:52:45  INFO  Ingestion date : 2026-03-10


2026-03-10 16:52:45  INFO  Raw data path  : /home/capinha/python_projects/data_engineering_sandbox/vasco_business_case/data/raw/RawData.csv


In [2]:
# I read 200KB of the file as raw bytes before decoding anything.Sampling is a common practice, since 
# chardet's confidence stabilises within the first KBs of a file so reading the whole file would not change the result.
# But would be slower on larger files in a real-world scenario.
with open(RAW_DATA_PATH, "rb") as f:
    raw_bytes = f.read(200_000)

# chardet looks for byte frequency patterns to figure out the encoding.
detection         = chardet.detect(raw_bytes)
detected_encoding = detection["encoding"]
confidence        = detection["confidence"]

logger.info("Detected encoding : %s (confidence %.1f%%)", detected_encoding, confidence * 100)

# Below 70% confidence I do not trust the result and I would probably try to get a fresh export.
if confidence < 0.7:
    logger.warning("Low confidence detected for encoding %s. Consider requesting a fresh export.", detected_encoding)

# In a past project I dealt with CSV files that had characters outside the set. We tried regex cleaning but 
# the compute cost to clean the amount of data we had was too high so we ended up requesting
# a fresh export. It was faster and cheaper than trying to fix corrupted character (not to mention easier)...
# For now I proceed with whatever encoding was detected since some data is better than none.

2026-03-10 16:52:45  INFO  Detected encoding : utf-8 (confidence 80.4%)


In [3]:
# I read the CSV file with the encoding detected by chardet. 'low_memory=False' makes pandas 
# read the full column before inferring its type. Without it, mixed-type columns on large files can trigger warnings.
df_raw = pd.read_csv(
    RAW_DATA_PATH,
    encoding=detected_encoding,
    low_memory=False,
)

# I tag each row so any record can always be traced back to the file and run that created it.
df_raw["_ingestion_date"] = INGESTION_DATE
df_raw["_source_file"]    = RAW_DATA_PATH.name

logger.info("Rows loaded : %s", f"{len(df_raw):,}")
logger.info("Columns     : %s", df_raw.shape[1])

# PRINT PREVIEW - I'll leave this in to make it easier to see the data. If you want to see the data, uncomment the line below.
#print(df_raw.head())

# Writing to Parquet is where the encoding fix happens automatically. Parquet stores all strings as UTF-8 internally so every downstream
# layer reads clean UTF-8 regardless of what the source encoding was.
raw_path = RAW_DIR / f"survey_raw_{INGESTION_DATE}.parquet"
df_raw.to_parquet(raw_path, index=False)

logger.info("Raw layer written : %s (%.1f KB)", raw_path, raw_path.stat().st_size / 1024)

2026-03-10 16:52:46  INFO  Rows loaded : 10,000


2026-03-10 16:52:46  INFO  Columns     : 131


2026-03-10 16:52:46  INFO  Raw layer written : ../data/raw_layer/survey_raw_2026-03-10.parquet (1170.2 KB)


In [4]:
# This is where the data is transformed. I create a copy so the raw layer dataframe is never touched by any transformation.
df_prepared = df_raw.copy()

# PRINT PREVIEW - I'll leave this in to make it easier to see the data. If you want to see the data, uncomment the line below.
#print(df_prepared.head())

logger.info("Starting row count : %s", f"{len(df_prepared):,}")


2026-03-10 16:52:46  INFO  Starting row count : 10,000


In [5]:
# The next couple of cells are where the actual logic to transform the data that will be in prepared layer is applied.
# Business Rule 1: respondents must be unique, keeping the first occurrence.

duplicates = df_prepared["Respondent"].duplicated().sum()
df_prepared = df_prepared.drop_duplicates(subset="Respondent", keep="first")

# Since it is expected to have one answer per person, duplicates are not expected.
# However, business should be notified if any are found so they can look into the root cause.
logger.info("Duplicate respondents removed: %s", f"{duplicates:,}")
if duplicates > 0:
    logger.warning("Duplicates found, please investigate")
logger.info("Rows after deduplication: %s", f"{len(df_prepared):,}")

2026-03-10 16:52:46  INFO  Duplicate respondents removed: 0


2026-03-10 16:52:46  INFO  Rows after deduplication: 10,000


In [6]:
# Business Rule 2: blank Student means No.
# I check for blanks and fill them with a "No".
student_nulls            = df_prepared["Student"].isna().sum()
df_prepared["Student"]   = df_prepared["Student"].fillna("No")

logger.info("Student nulls defaulted to No: %s", f"{student_nulls:,}")

# Business Rule 3: blank Employment means full-time. Same procedure as above.
employment_nulls          = df_prepared["Employment"].isna().sum()
df_prepared["Employment"] = df_prepared["Employment"].fillna("Employed full-time")

logger.info("Employment nulls defaulted to Employed full-time: %s", f"{employment_nulls:,}")

2026-03-10 16:52:46  INFO  Student nulls defaulted to No: 232


2026-03-10 16:52:46  INFO  Employment nulls defaulted to Employed full-time: 189


In [7]:
# Business Rule 4: responses with more than 3 empty fields are considered invalid.
# Pipeline metadata columns that I createdare excluded since they are not survey fields.
survey_cols = [c for c in df_prepared.columns if not c.startswith("_")]

# When I started this, the amount of rows that were going to be dropped was huge.
# So, instead of dropping rows, I added three new columns: 'empty_fields', 'empty_field_count' and 'more_than_three_empty'.
# -> 'more_than_three_empty' is the direct expression of the business rule, derived from the count. 
#     The business can use it to filter easily for the rows that do not pass the original rule.
# -> 'empty_field_count' is the count of empty fields for each row, it can be used to better analyse the amount of empty fields.
# -> 'empty_fields' is a list of the column names that are null for each row. 
#     If the same fields keep appearing across many rows it may point to a question that respondents consistently skip.
#     Making this a list allows the business use SQL the fields that are empty for each row, making it easier to spot patterns.

df_prepared["empty_fields"] = df_prepared[survey_cols].apply(
    lambda row: [col for col in survey_cols if pd.isna(row[col])], axis=1)

df_prepared["empty_field_count"] = df_prepared["empty_fields"].apply(len)


df_prepared["more_than_three_empty"] = df_prepared["empty_field_count"] > 3

flagged = df_prepared["more_than_three_empty"].sum()
logger.info("Rows flagged as more than three empty : %s of %s (%.1f%%)",
            f"{flagged:,}", f"{len(df_prepared):,}", flagged / len(df_prepared) * 100)
logger.info("Rows that pass the rule               : %s",
            f"{(~df_prepared['more_than_three_empty']).sum():,}")

2026-03-10 16:52:47  INFO  Rows flagged as more than three empty : 9,603 of 10,000 (96.0%)


2026-03-10 16:52:47  INFO  Rows that pass the rule               : 397


In [8]:
# Business Rule 5: all salaries should be yearly.
# I create 'salary_yearly' as a new column and keep the original Salary and SalaryType so the raw values are always there if needed.

def clean_salary_str(s):
    # Salary strings can use a comma as a decimal separator (European style, e.g. "981,51")
    # or as a thousands separator (e.g. "22,000.00"). I distinguish them by checking
    # whether a dot is already present and whether the string ends with 1-2 digits after the last comma.
    if re.search(r',\d{1,2}$', s) and '.' not in s:
        return re.sub(r',(?=\d{1,2}$)', '.', s).replace(',', '')
    return s.replace(',', '')

def normalize_salary(row):
    salary = row["Salary"]
    stype  = row["SalaryType"]
    if pd.isna(salary):
        return None
    salary = float(clean_salary_str(str(salary)))  # Salary arrives as a string from the CSV; without this cast, multiplying a string repeats it instead of doing arithmetic.
    if stype == "Weekly":
        return salary * 52
    if stype == "Monthly":
        return salary * 12
    if stype == "Yearly":
        return salary
    # When SalaryType is null with a salary value present, I decided to discard it, since I don't know if the value makes sense as a yearly salary.
    return None

# I count the rows I will discard before applying it so I can know how many rows I will discard.
null_type_with_salary = df_prepared[df_prepared["Salary"].notna() & df_prepared["SalaryType"].isna()].shape[0]

df_prepared["salary_yearly"] = df_prepared.apply(normalize_salary, axis=1)

logger.info("Rows discarded due to null SalaryType with a salary value : %s", f"{null_type_with_salary:,}")
logger.info("Rows with a yearly salary value                           : %s", f"{df_prepared['salary_yearly'].notna().sum():,}")
logger.info("Rows without salary data                                  : %s", f"{df_prepared['salary_yearly'].isna().sum():,}")

2026-03-10 16:52:48  INFO  Rows discarded due to null SalaryType with a salary value : 1,025


2026-03-10 16:52:48  INFO  Rows with a yearly salary value                           : 6,381


2026-03-10 16:52:48  INFO  Rows without salary data                                  : 3,619


In [9]:
# After all the transformations, I write the actual Prepared layer.
# Downstream consumers read from here rather than from the raw CSV.
# Azure: path points to the prepared_layer container in ADLS Gen2, partitioned by date.
prepared_path = PREPARED_DIR / f"survey_prepared_{INGESTION_DATE}.parquet"
df_prepared.to_parquet(prepared_path, index=False)

# PRINT PREVIEW - I'll leave this in to make it easier to see the data. If you want to see the data, uncomment the line below.
#print(df_prepared.head())

logger.info("Prepared layer written : %s (%.1f KB)", prepared_path, prepared_path.stat().st_size / 1024)
logger.info("Row count              : %s", f"{len(df_prepared):,}")
logger.info("Column count           : %s", df_prepared.shape[1])

2026-03-10 16:52:48  INFO  Prepared layer written : ../data/prepared_layer/survey_prepared_2026-03-10.parquet (1315.1 KB)


2026-03-10 16:52:48  INFO  Row count              : 10,000


2026-03-10 16:52:48  INFO  Column count           : 135


In [10]:
# The Staging layer will contain only information that the business asked for. This is the minimum amount of information
# that the business needs to answer the questions they asked for.
# -> Analysis 1: which survey questions have the highest variance in answers across participants.
# -> Analysis 2: whether there is a relationship between years of coding experience and salary.
# I will also keep demographic columns so both analyses can be broken down by participant if needed. 
# The full dataset is available in the Prepared layer. If business needs more columns, they can be easily added
# to the Staging layer via the pipeline.

RENAME_MAP = {
    "Respondent"         : "respondent_id",
    "Country"            : "country",
    "Student"            : "student",
    "Employment"         : "employment",
    "FormalEducation"    : "formal_education",
    "UndergradMajor"     : "undergrad_major",
    "CompanySize"        : "company_size",
    "DevType"            : "dev_type",
    "YearsCoding"        : "years_coding",
    "YearsCodingProf"    : "years_coding_prof",
    "JobSatisfaction"    : "job_satisfaction",
    "CareerSatisfaction" : "career_satisfaction",
    "Salary"             : "salary_original",
    "SalaryType"         : "salary_type",
    "ConvertedSalary"    : "converted_salary_usd",
    "Currency"           : "currency",
    "CurrencySymbol"     : "currency_symbol",
    "Gender"             : "gender",
    "Age"                : "age",
}

# I rename the columns to snake_case since it makes it easier to work with them in SQL.

df_staging = df_prepared.rename(columns=RENAME_MAP)

# Like I mentioned before, the flag columns created for the prepared layer are included so analysts can filter them out as needed.
STAGING_COLS = [
    "respondent_id", "country", "student", "employment",
    "formal_education", "undergrad_major", "company_size", "dev_type",
    "years_coding", "years_coding_prof",
    "job_satisfaction", "career_satisfaction",
    "salary_original", "salary_type", "salary_yearly", "converted_salary_usd",
    "currency", "currency_symbol",
    "gender", "age",
    "empty_field_count", "more_than_three_empty",
    "_ingestion_date", "_source_file",
]

df_staging = df_staging[[c for c in STAGING_COLS if c in df_staging.columns]]

# Azure: writing as Delta format instead of Parquet here would allow Fabric/Synapse
# to expose this table automatically via a SQL endpoint with no extra setup.
# df_staging.write.format("delta").save(staging_path)  # PySpark equivalent on Databricks/Fabric
staging_path = STAGING_DIR / f"fact_survey_{INGESTION_DATE}.parquet"
df_staging.to_parquet(staging_path, index=False)

# PRINT PREVIEW - I'll leave this in to make it easier to see the data. If you want to see the data, uncomment the line below.
#print(df_staging.head())

logger.info("Staging layer written : %s (%.1f KB)", staging_path, staging_path.stat().st_size / 1024)
logger.info("Rows %s  Columns %s", f"{len(df_staging):,}", df_staging.shape[1])


2026-03-10 16:52:48  INFO  Staging layer written : ../data/staging_layer/fact_survey_2026-03-10.parquet (271.2 KB)


2026-03-10 16:52:48  INFO  Rows 10,000  Columns 24


In [11]:
# Analysis 1: variance per survey question.
# df_prepared is still in memory so no extra read from disk is needed. I use it to ensure the calculations are considering all rows.

# I grab all numeric columns to start with.
numeric_cols = df_prepared.select_dtypes(include="number").columns.tolist()

# Pipeline and flag columns are not survey questions so I remove them.
exclude      = {"empty_field_count", "more_than_three_empty"}
question_cols = [c for c in numeric_cols if c not in exclude]

# I calculate the variance across all respondents and sort them from highest to lowest.
variance_df = (
    df_prepared[question_cols]
    .var()
    .reset_index()
    .rename(columns={"index": "question", 0: "variance"})
    .sort_values("variance", ascending=False)
    .reset_index(drop=True)
)

# Azure: same Delta format note as the staging table above applies here.
variance_path = STAGING_DIR / f"fact_variance_{INGESTION_DATE}.parquet"
variance_df.to_parquet(variance_path, index=False)

logger.info("Variance table written : %s", variance_path)
logger.info("Top 10 questions by variance:")
for _, row in variance_df.head(10).iterrows():
    logger.info("  %-30s  %.2f", row["question"], row["variance"])

2026-03-10 16:52:48  INFO  Variance table written : ../data/staging_layer/fact_variance_2026-03-10.parquet


2026-03-10 16:52:48  INFO  Top 10 questions by variance:


2026-03-10 16:52:48  INFO    salary_yearly                   189970729704946888359577596546832681657944257201530334385130344032901309229691233139865562444759880759600016910492457375589307311448416268767810610528256.00


2026-03-10 16:52:48  INFO    ConvertedSalary                 40885354284.07


2026-03-10 16:52:48  INFO    Respondent                      863848800.53


...

2026-03-10 16:52:48  INFO    AssessBenefits6                 8.00


2026-03-10 16:52:48  INFO    AssessBenefits3                 7.97


2026-03-10 16:52:48  INFO    AssessBenefits4                 7.85


2026-03-10 16:52:48  INFO    AssessBenefits8                 7.73


2026-03-10 16:52:48  INFO    AssessJob1                      7.72


In [12]:
# The output above shows ConvertedSalary and Respondent at the top with variance in the billions.
# This makes sense since they are a continuous monetary value and a row identifier, not a normal survey question.
# So I'll remove them. In my opinion, it only makes sense to keep the actual survey ranking questions where looking at the variance matters.

# Once the business confirms this is the right approach, these columns will be removed from the previous cell
# to save compute on every pipeline run. I just left it here to remind myself to mention it.

non_question_cols   = {"ConvertedSalary", "Respondent", "salary_yearly"}
question_cols_clean = [c for c in question_cols if c not in non_question_cols]

variance_df_clean = (
    df_prepared[question_cols_clean]
    .var()
    .reset_index()
    .rename(columns={"index": "question", 0: "variance"})
    .sort_values("variance", ascending=False)
    .reset_index(drop=True)
)

# I overwrite the variance file with this cleaned version.
variance_df_clean.to_parquet(variance_path, index=False)

logger.info("Top 10 survey questions by variance after removing non-question columns:")
for _, row in variance_df_clean.head(10).iterrows():
    logger.info("  %-30s  %.2f", row["question"], row["variance"])

2026-03-10 16:52:48  INFO  Top 10 survey questions by variance after removing non-question columns:


2026-03-10 16:52:48  INFO    AssessBenefits2                 9.35


2026-03-10 16:52:48  INFO    AssessJob7                      8.39


2026-03-10 16:52:48  INFO    AssessBenefits6                 8.00


2026-03-10 16:52:48  INFO    AssessBenefits3                 7.97


...

2026-03-10 16:52:48  INFO    AssessBenefits8                 7.73


2026-03-10 16:52:48  INFO    AssessJob1                      7.72


2026-03-10 16:52:48  INFO    AssessBenefits9                 7.58


2026-03-10 16:52:48  INFO    AssessBenefits7                 7.39


2026-03-10 16:52:48  INFO    AssessBenefits11                7.21


In [13]:
# Analysis 2: relationship between years of coding experience and salary.

# YearsCoding is stored as a text range like "6-8 years" so I map everything to the value in the middle of each range, 
# to make it easier to calculate the correlation.
YEARS_MAP = {
    "0-2 years"        : 1,
    "3-5 years"        : 4,
    "6-8 years"        : 7,
    "9-11 years"       : 10,
    "12-14 years"      : 13,
    "15-17 years"      : 16,
    "18-20 years"      : 19,
    "21-23 years"      : 22,
    "24-26 years"      : 25,
    "27-29 years"      : 28,
    "30 or more years" : 32,
}

df_prepared["years_coding_numeric"] = df_prepared["YearsCoding"].map(YEARS_MAP)

# I use ConvertedSalary rather than salary_yearly because it ensures I'm comparing apples to apples.
# The column salary_yearly that I created earlier keeps values in the participant's currency which would skew the correlation.
salary_years = df_prepared[["years_coding_numeric", "ConvertedSalary"]].dropna()

correlation = salary_years["years_coding_numeric"].corr(salary_years["ConvertedSalary"])

logger.info("Rows used for correlation                    : %s", f"{len(salary_years):,}")
logger.info("Correlation (YearsCoding vs ConvertedSalary) : %.4f", correlation)
logger.info("%s",
    "The results show that more experience tends to mean higher salary" if correlation > 0
    else "The results show that more experience tends to mean lower salary")

2026-03-10 16:52:48  INFO  Rows used for correlation                    : 7,244


2026-03-10 16:52:48  INFO  Correlation (YearsCoding vs ConvertedSalary) : 0.1536


2026-03-10 16:52:48  INFO  The results show that more experience tends to mean higher salary
